In [ ]:
# =====================================================
# INPAINTING: AE vs VAE vs U-NET
# =====================================================
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 5
LATENT_DIM = 128

# =====================================================
# DATASET
# =====================================================
dataset = tfds.load("lfw", split="train", data_dir="data")

In [ ]:
def preprocess(example):
    image = example["image"]
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image

In [ ]:
# =====================================================
# МАСКА (быстрая)
# =====================================================
def mask_image(image):
    mask_size = 16

    h = tf.shape(image)[0]
    w = tf.shape(image)[1]

    x = tf.random.uniform([], 0, w - mask_size, dtype=tf.int32)
    y = tf.random.uniform([], 0, h - mask_size, dtype=tf.int32)

    mask = tf.ones_like(image)

    zeros = tf.zeros((mask_size, mask_size, 3))
    paddings = [[y, h - y - mask_size],
                [x, w - x - mask_size],
                [0, 0]]

    mask_patch = tf.pad(zeros, paddings, constant_values=1.0)

    masked = image * mask_patch

    return masked, image

dataset = dataset.map(preprocess)
dataset = dataset.map(mask_image)
dataset = dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# =====================================================
# VISUALIZATION FUNCTION
# =====================================================
def show_results(model, dataset, title):
    masked, original = next(iter(dataset))
    pred = model.predict(masked[:5])

    for i in range(5):
        plt.figure(figsize=(9,3))
        plt.suptitle(title)

        plt.subplot(1,3,1)
        plt.imshow(masked[i])
        plt.title("Masked")

        plt.subplot(1,3,2)
        plt.imshow(pred[i])
        plt.title("Reconstructed")

        plt.subplot(1,3,3)
        plt.imshow(original[i])
        plt.title("Original")

        plt.show()

In [ ]:
# =====================================================
# 1. AUTOENCODER (AE)
# =====================================================
def build_ae():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    # Encoder
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)

    # Decoder
    x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)

    outputs = layers.Conv2D(3, 3, activation='sigmoid', padding='same')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')

    return model

In [ ]:
# ------------------------------
# Sampling layer
# ------------------------------
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps

# ------------------------------
# Build VAE
# ------------------------------
def build_vae():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    # Encoder
    x = layers.Conv2D(32, 3, activation='relu', padding='same', strides=2)(inputs)
    x = layers.Conv2D(64, 3, activation='relu', padding='same', strides=2)(x)
    x = layers.Flatten()(x)
    z_mean = layers.Dense(LATENT_DIM)(x)
    z_log_var = layers.Dense(LATENT_DIM)(x)
    z = Sampling()([z_mean, z_log_var])

    # Decoder
    x = layers.Dense((IMG_SIZE//4)*(IMG_SIZE//4)*64, activation='relu')(z)
    x = layers.Reshape((IMG_SIZE//4, IMG_SIZE//4, 64))(x)
    x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
    outputs = layers.Conv2D(3, 3, activation='sigmoid', padding='same')(x)

    # ------------------------------
    # Custom VAE Model
    # ------------------------------
    class VAE(Model):
        def __init__(self, encoder_model, decoder_model):
            super().__init__()
            self.encoder = encoder_model
            self.decoder = decoder_model

        def call(self, inputs):
            z_mean, z_log_var, z = self.encoder(inputs)
            return self.decoder(z)

        def train_step(self, data):
            x, y = data  # batch вход и цель

            with tf.GradientTape() as tape:
                # получаем реальные тензоры z_mean, z_log_var через энкодер
                z_mean, z_log_var, z = self.encoder(x, training=True)
                reconstruction = self.decoder(z, training=True)

                # Reconstruction loss
                rec_loss = tf.reduce_mean(tf.reduce_sum(tf.square(y - reconstruction), axis=[1,2,3]))

                # KL divergence
                kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1
                ))

                loss = rec_loss + kl_loss

            grads = tape.gradient(loss, self.trainable_weights)
            self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

            return {"loss": loss, "rec_loss": rec_loss, "kl_loss": kl_loss}

    # Создаем encoder и decoder модели
    encoder_model = Model(inputs, [z_mean, z_log_var, z], name="encoder")
    decoder_input = layers.Input(shape=(LATENT_DIM,))
    x_dec = layers.Dense((IMG_SIZE//4)*(IMG_SIZE//4)*64, activation='relu')(decoder_input)
    x_dec = layers.Reshape((IMG_SIZE//4, IMG_SIZE//4, 64))(x_dec)
    x_dec = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x_dec)
    x_dec = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x_dec)
    decoder_output = layers.Conv2D(3, 3, activation='sigmoid', padding='same')(x_dec)
    decoder_model = Model(decoder_input, decoder_output, name="decoder")

    vae = VAE(encoder_model, decoder_model)
    vae.compile(optimizer='adam')

    return vae

In [ ]:
# =====================================================
# 3. U-NET
# =====================================================
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

def build_unet():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    # bottleneck
    bn = conv_block(p2, 128)

    # decoder
    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(bn)
    u1 = layers.Concatenate()([u1, c2])
    c3 = conv_block(u1, 64)

    u2 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(c3)
    u2 = layers.Concatenate()([u2, c1])
    c4 = conv_block(u2, 32)

    outputs = layers.Conv2D(3, 1, activation='sigmoid')(c4)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')

    return model


In [ ]:
import datetime

In [ ]:
# =====================================================
# ОБУЧЕНИЕ
# =====================================================

In [ ]:
print("Training AE...")
ae = build_ae()
ae.summary()

In [ ]:
log_dir = "logs/autoencoder/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
ae.fit(dataset, epochs=EPOCHS, shuffle=True,callbacks=[tensorboard_callback])

In [ ]:
show_results(ae, dataset, "Autoencoder")

In [ ]:
print("Training VAE...")
vae = build_vae()
vae.summary()

In [ ]:
log_dir = "logs/var_autoencoder/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
vae.fit(dataset, epochs=2, shuffle=True,callbacks=[tensorboard_callback])

In [ ]:
show_results(vae, dataset, "VAE")

In [ ]:
print("Training U-Net...")
unet = build_unet()
unet.summary()

In [ ]:
log_dir = "logs/unet/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
unet.fit(dataset, epochs=EPOCHS, shuffle=True,callbacks=[tensorboard_callback])

In [ ]:
show_results(unet, dataset, "U-Net")